In [11]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import operator

In [3]:
ticker_file = pd.read_csv('Tickers_Example.csv', header=None)
ticker_file.rename(columns={0: 'NAME'}, inplace=True)

In [4]:
def clean_data(tickers):
    filtered_stocks = pd.DataFrame()
    start_date = "2023-10-01"
    end_date = "2024-09-30"
    for ticker in tickers['NAME']:
        try:
            stock = yf.Ticker(ticker)
            info = stock.fast_info
            if info['currency'] not in ["USD", "CAD"]: # Enusring stock is listed, traded in CAD or USD
                continue
    
            hist = stock.history(start=start_date, end=end_date, interval="1d")
            hist['Month'] = hist.index.to_period('M')
            monthly_data = hist.groupby('Month').filter(lambda x: len(x) >= 18)

            avg_monthly_volume = monthly_data.groupby('Month')['Volume'].mean().mean()
            if avg_monthly_volume >= 100000:
                filtered_stocks = pd.concat([filtered_stocks, pd.DataFrame({"Ticker": [ticker]})])
                
        except Exception as e:
            continue
    
    return filtered_stocks.reset_index(drop=True)

In [5]:
ticker_file = clean_data(ticker_file)

/var/folders/7x/z5l3v4x13gv5wlh0qdjwgllh0000gn/T/ipykernel_55689/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
/var/folders/7x/z5l3v4x13gv5wlh0qdjwgllh0000gn/T/ipykernel_55689/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
/var/folders/7x/z5l3v4x13gv5wlh0qdjwgllh0000gn/T/ipykernel_55689/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
/var/folders/7x/z5l3v4x13gv5wlh0qdjwgllh0000gn/T/ipykernel_55689/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
$AGN: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
/var/folde

In [6]:
def getBeta(cur_ticker):
    SP_ticker = '^GSPC'

    cur_stock = yf.Ticker(cur_ticker)
    SP_index = yf.Ticker(SP_ticker)

    # Will edit these later
    start_date = '2022-11-14'
    end_date = '2024-11-14'

    cur_stock_hist = cur_stock.history(start=start_date, end=end_date)
    SP_index_hist = SP_index.history(start=start_date, end=end_date)

    prices = pd.DataFrame()
    prices[cur_ticker] = pd.DataFrame(cur_stock_hist['Close'])
    prices['S&P Index'] = SP_index_hist['Close']

    daily_returns = prices.pct_change(fill_method=None).dropna()
    daily_returns.drop(index=daily_returns.index[0], inplace=True)

    SP_var = daily_returns['S&P Index'].var()
    SP_beta = daily_returns.cov() / SP_var

    return SP_beta.iat[0,1]

In [7]:
# Gets average of daily growth
def getGrowth(cur_ticker):
    # Fetch data for current stock
    cur_stock = yf.Ticker(cur_ticker)
    start_date = '2022-11-14'
    end_date = '2024-11-14'
    cur_stock_hist = cur_stock.history(start=start_date, end=end_date)

    prices = pd.DataFrame()
    prices[cur_ticker] = cur_stock_hist['Close']
    
    daily_returns = prices.pct_change(fill_method=None).dropna()
    SP_growth = daily_returns.mean()
    
    return SP_growth[cur_ticker] * 100

In [8]:
def getVolatility(cur_ticker):
    start_date = '2022-11-14'
    end_date = '2024-11-14'
    stock_data = yf.Ticker(cur_ticker).history(start=start_date, end=end_date)
    stock_data['Daily Return'] = stock_data['Close'].pct_change(fill_method=None).dropna()
    
    volatility = stock_data['Daily Return'].std()
    return volatility

In [ ]:
stock_data = []
beta_data = []
volatility_data =[]
growth_data =[]

for i in range(len(ticker_file)):
    cur_ticker = ticker_file['Ticker'].iloc[i]

    beta = getBeta(cur_ticker)
    beta_data += [{ "name": cur_ticker, "beta": beta }]
    growth = getGrowth(cur_ticker)
    growth_data += [{ "name": cur_ticker, "growth": growth }]
    volatility = getVolatility(cur_ticker)
    volatility_data += [{ "name": cur_ticker, "volatility": volatility }]

    stock_details = {
        "name": cur_ticker,
        "beta": beta,
        "growth": growth,
        "volatility": volatility
    }

    stock_data = stock_data + [stock_details]

sorted_betas = sorted(beta_data, key=lambda d: d['beta'], reverse=True)
sorted_growth = sorted(growth_data, key=lambda d: d['growth'], reverse=True)
sorted_volatility = sorted(volatility_data, key=lambda d: d['volatility'], reverse=True)

#for i in range(len(stock_data)):
#    print(stock_data[i])

[{'name': 'SHOP.TO', 'volatility': 0.03470011598895072},
 {'name': 'BB.TO', 'volatility': 0.03411414832944686},
 {'name': 'QCOM', 'volatility': 0.022530974370406222},
 {'name': 'PYPL', 'volatility': 0.02236705416806987},
 {'name': 'USB', 'volatility': 0.02114484058058287},
 {'name': 'BA', 'volatility': 0.019634197749691013},
 {'name': 'AMZN', 'volatility': 0.01947138887810965},
 {'name': 'LLY', 'volatility': 0.018204258182402656},
 {'name': 'CAT', 'volatility': 0.017216754939850882},
 {'name': 'C', 'volatility': 0.01645916109180432},
 {'name': 'TXN', 'volatility': 0.01634078160869916},
 {'name': 'UPS', 'volatility': 0.01599797410455374},
 {'name': 'BAC', 'volatility': 0.015906184112643725},
 {'name': 'AXP', 'volatility': 0.01579194511544664},
 {'name': 'BMY', 'volatility': 0.01514068517518797},
 {'name': 'BIIB', 'volatility': 0.015030732386723335},
 {'name': 'ACN', 'volatility': 0.014994615338540226},
 {'name': 'AIG', 'volatility': 0.014746090689415122},
 {'name': 'UNH', 'volatility': 